# MMIT Claim-Level EDA Checks

Claim-level validation of MMIT matching, feature-table coverage, Bayer HPM alignment, and univariate lift for Kerendia claims using the existing access-join logic plus winning `OrgId` / `FormularyId` retention for downstream feature joins.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
from pyspark.sql import functions as F, Window
import pandas as pd
import numpy as np

catalog = "`ph-com-ai-us-prod-catalog-2553213839599380`"
LOOKBACK_START = "2024-09-01"
LOOKBACK_END = "2026-08-31"
COVERED_START = 202510
COVERED_END = 202608
SECTION4_START = 202601
SECTION4_END = 202608
SECTION5_END = 202607
KERENDIA_MEDID = 607231
JARDIANCE_MEDIDS = [582254, 582255]
FARXIGA_MEDID = 580683
INVOKANA_MEDID = 577710
STEGLATRO_MEDID = 595221
TOP3_PBMS = ["CVS Caremark RX", "Express Scripts PBM", "OptumRx"]


def pdf_to_md_table(pdf: pd.DataFrame) -> str:
    if pdf.empty:
        return "| No rows |\n| --- |"
    pdf = pdf.copy()
    pdf.columns = [str(c) for c in pdf.columns]
    for c in pdf.columns:
        pdf[c] = pdf[c].map(lambda x: "" if pd.isna(x) else str(x))
    header = "| " + " | ".join(pdf.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(pdf.columns)) + " |"
    rows = ["| " + " | ".join(map(str, row)) + " |" for row in pdf.itertuples(index=False, name=None)]
    return "\n".join([header, sep] + rows)


def pct(n, d):
    return None if d in (0, None) else 100.0 * float(n) / float(d)


def month_to_date(col_name: str):
    return F.to_date(F.concat(F.col(col_name).cast("string"), F.lit("01")), "yyyyMMdd")


def access_category(col_name: str):
    return (
        F.when(F.col(col_name).isin("Preferred", "Covered", "Generic (Preferred)"), "Covered")
        .when(
            F.col(col_name).isin(
                "Covered (PA/ST)", "Preferred (PA/ST)",
                "Generic (Preferred) (PA/ST)", "Generic (PA/ST)"
            ),
            "Covered_with_PA_ST"
        )
        .when(F.col(col_name) == "Specialty", "Specialty")
        .when(F.col(col_name) == "Specialty (PA/ST)", "Specialty_with_PA_ST")
        .when(F.col(col_name) == "Not Covered", "Not_Covered")
        .when(F.col(col_name) == "Not Listed", "Not_Listed")
        .when(F.col(col_name).isNull(), "No_MMIT_Match")
        .otherwise("Unknown")
    )


status_rank_expr = (
    F.when(F.col("status_rollup").isin("Preferred", "Generic (Preferred)"), F.lit(6))
    .when(F.col("status_rollup") == "Covered", F.lit(5))
    .when(F.col("status_rollup").isin("Preferred (PA/ST)", "Specialty"), F.lit(4))
    .when(F.col("status_rollup").isin("Covered (PA/ST)", "Generic (Preferred) (PA/ST)", "Generic (PA/ST)"), F.lit(3))
    .when(F.col("status_rollup") == "Specialty (PA/ST)", F.lit(2))
    .when(F.col("status_rollup") == "Not Listed", F.lit(1))
    .when(F.col("status_rollup") == "Not Covered", F.lit(0))
)

print("Helpers loaded.")

In [0]:
rx_query = f"""
    SELECT DISTINCT
        A.CLAIM_ID,
        A.PROVIDER_ID,
        A.PATIENT_ID,
        A.SVC_DT,
        A.DAYS_SUPPLY,
        A.NDC_CD,
        C.PRODUCT_GROUP,
        C.PRODUCT_STRENGTH,
        C.GENERIC_NAME,
        A.REFILL_CODE,
        A.REJECT_CODE,
        A.PAYER_PLAN_ID,
        A.SOB,
        A.CLAIM_TYPE,
        A.DAW_CODE,
        A.ZIP_CODE,
        A.CHANNEL_CODE,
        A.CLAIM_STATUS,
        A.COPAY_CARD_FLG
    FROM (
        SELECT CLAIM_ID, PATIENT_ID, SVC_DT, DAYS_SUPPLY, QUANTITY,
               REFILL_CODE, REJECT_CODE, PROVIDER_ID, NDC_CD,
               PAYER_PLAN_ID, SOB, CLAIM_TYPE, DAW_CODE, ZIP_CODE,
               CHANNEL_CODE, CLAIM_STATUS, COPAY_CARD_FLG
        FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX
    ) A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD C
        ON A.NDC_CD = C.NDC_CD
    WHERE A.PROVIDER_ID IS NOT NULL
      AND C.PRODUCT_GROUP = 'KERENDIA'
      AND A.SVC_DT >= '{LOOKBACK_START}'
      AND A.SVC_DT <= '{LOOKBACK_END}'
"""

rx_claims_raw = Get_Data_Snowflakes(rx_query)
kerendia_claims = (
    rx_claims_raw
    .withColumn("claim_month", F.date_format("SVC_DT", "yyyyMM").cast("int"))
    .withColumn("PAYER_PLAN_ID_LONG", F.col("PAYER_PLAN_ID").cast("long"))
    .select(
        "CLAIM_ID", "PROVIDER_ID", "PATIENT_ID", "SVC_DT", "claim_month",
        "PAYER_PLAN_ID", "PAYER_PLAN_ID_LONG", "CLAIM_STATUS", "CLAIM_TYPE"
    )
    .dropDuplicates(["CLAIM_ID"])
    .cache()
)
_ = kerendia_claims.count()

print(f"Claims cached: {kerendia_claims.count():,}")
print(f"Distinct HCPs: {kerendia_claims.select('PROVIDER_ID').distinct().count():,}")
print(f"Distinct patients: {kerendia_claims.select('PATIENT_ID').distinct().count():,}")
print(f"Distinct payer plans: {kerendia_claims.select('PAYER_PLAN_ID').distinct().count():,}")

display(
    kerendia_claims.groupBy("claim_month")
    .agg(F.count("*").alias("claims"), F.countDistinct("PROVIDER_ID").alias("hcps"))
    .orderBy("claim_month")
)

display(
    kerendia_claims.groupBy("CLAIM_TYPE", "CLAIM_STATUS")
    .count()
    .orderBy(F.desc("count"))
    .limit(12)
)

In [0]:
target_med_ids = [INVOKANA_MEDID, FARXIGA_MEDID] + JARDIANCE_MEDIDS + [KERENDIA_MEDID]
med_to_drug = {
    INVOKANA_MEDID: "invokana",
    FARXIGA_MEDID: "farxiga",
    582254: "jardiance",
    582255: "jardiance",
    KERENDIA_MEDID: "kerendia",
}

bridge = spark.sql(f"""
    SELECT DISTINCT OrgId, IMSPayerPlanId
    FROM {catalog}.market_access.bronze_iqvia_bridge
""")

plan_dim = spark.sql(f"""
    SELECT DISTINCT OrgId, FormularyId, Channel, PlanType, PBMName, BenefitDesign,
           NotListedPolicy, NonFormularyPolicy, PBMRelationship, PlanName
    FROM {catalog}.market_access.silver_plan_dim
    WHERE is_current = true
""")

lives_per_org = spark.sql(f"""
    SELECT OrgId, Period, SUM(Lives) AS total_lives
    FROM {catalog}.market_access.bronze_states
    WHERE Period BETWEEN {COVERED_START} AND 202609
    GROUP BY OrgId, Period
""")

status_raw = spark.sql(f"""
    SELECT FormularyId, MedId, Period, TherapeuticClass, UniversalStatusRollup, RawStatus, feed_date
    FROM {catalog}.market_access.bronze_statuses
    WHERE MedId IN ({','.join(map(str, target_med_ids))})
      AND Period BETWEEN {COVERED_START} AND 202609
""")

latest_status_window = Window.partitionBy("FormularyId", "MedId", "Period", "TherapeuticClass").orderBy(F.col("feed_date").desc())
status_snapshot = (
    status_raw
    .withColumn("rn", F.row_number().over(latest_status_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

med_mapping = spark.createDataFrame([(k, v) for k, v in med_to_drug.items()], ["MedId", "drug_label"])
status_with_drug = status_snapshot.join(med_mapping, on="MedId", how="inner")

status_priority = {
    "Preferred": 1, "Covered": 2, "Generic (Preferred)": 3, "Specialty": 4,
    "Covered (PA/ST)": 5, "Preferred (PA/ST)": 6, "Generic (Preferred) (PA/ST)": 7,
    "Generic (PA/ST)": 8, "Specialty (PA/ST)": 9, "Not Listed": 10, "Not Covered": 11
}
priority_mapping = spark.createDataFrame([(k, v) for k, v in status_priority.items()], ["UniversalStatusRollup", "status_priority"])

status_deduped = (
    status_with_drug
    .join(priority_mapping, on="UniversalStatusRollup", how="left")
    .withColumn("status_priority", F.coalesce(F.col("status_priority"), F.lit(99)))
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("FormularyId", "drug_label", "Period").orderBy("status_priority", "RawStatus")
        )
    )
    .filter(F.col("rn") == 1)
    .select(
        "FormularyId", "drug_label", "Period",
        F.col("UniversalStatusRollup").alias("status_rollup"),
        F.col("RawStatus").alias("raw_status"),
        "TherapeuticClass"
    )
)

payer_formulary_with_lives = (
    bridge
    .join(plan_dim, on="OrgId", how="inner")
    .join(lives_per_org, on="OrgId", how="left")
)

payer_drug_status = (
    payer_formulary_with_lives
    .join(status_deduped, on=["FormularyId", "Period"], how="inner")
    .select(
        "IMSPayerPlanId", "OrgId", "FormularyId", "Period", "drug_label",
        "status_rollup", "raw_status", "total_lives",
        "Channel", "PlanType", "PBMName", "BenefitDesign",
        "NotListedPolicy", "NonFormularyPolicy", "PBMRelationship", "PlanName"
    )
)

winner_window = Window.partitionBy("IMSPayerPlanId", "Period", "drug_label").orderBy(F.col("total_lives").desc_nulls_last(), F.col("OrgId"), F.col("FormularyId"))
payer_status_consolidated = (
    payer_drug_status
    .withColumn("rn", F.row_number().over(winner_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

kerendia_winner = (
    payer_status_consolidated
    .filter(F.col("drug_label") == "kerendia")
    .select(
        "IMSPayerPlanId", "Period",
        F.col("OrgId").alias("winning_OrgId"),
        F.col("FormularyId").alias("winning_FormularyId"),
        F.col("total_lives").alias("winning_total_lives"),
        "Channel", "PlanType", "PBMName", "BenefitDesign",
        "NotListedPolicy", "NonFormularyPolicy", "PBMRelationship", "PlanName",
        F.col("status_rollup").alias("kerendia_status"),
        F.col("raw_status").alias("kerendia_raw_status")
    )
)

payer_status_pivoted = (
    payer_status_consolidated
    .groupBy("IMSPayerPlanId", "Period")
    .agg(
        F.first(F.when(F.col("drug_label") == "kerendia", F.col("status_rollup")), ignorenulls=True).alias("kerendia_status"),
        F.first(F.when(F.col("drug_label") == "jardiance", F.col("status_rollup")), ignorenulls=True).alias("jardiance_status"),
        F.first(F.when(F.col("drug_label") == "farxiga", F.col("status_rollup")), ignorenulls=True).alias("farxiga_status"),
        F.first(F.when(F.col("drug_label") == "invokana", F.col("status_rollup")), ignorenulls=True).alias("invokana_status"),
        F.first(F.when(F.col("drug_label") == "kerendia", F.col("raw_status")), ignorenulls=True).alias("kerendia_raw_status"),
        F.first(F.when(F.col("drug_label") == "jardiance", F.col("raw_status")), ignorenulls=True).alias("jardiance_raw_status"),
        F.first(F.when(F.col("drug_label") == "farxiga", F.col("raw_status")), ignorenulls=True).alias("farxiga_raw_status"),
        F.first(F.when(F.col("drug_label") == "invokana", F.col("raw_status")), ignorenulls=True).alias("invokana_raw_status")
    )
    .join(kerendia_winner.drop("kerendia_status", "kerendia_raw_status"), on=["IMSPayerPlanId", "Period"], how="left")
)

kerendia_periods = (
    status_deduped.filter(F.col("drug_label") == "kerendia")
    .select(F.col("Period").alias("period_key"))
    .distinct()
)

payer_period_plan = (
    payer_formulary_with_lives
    .filter(F.col("Period").between(COVERED_START, COVERED_END))
    .select("IMSPayerPlanId", "Period")
    .distinct()
    .withColumnRenamed("Period", "plan_period")
)

claims_with_access = (
    kerendia_claims
    .join(kerendia_periods, kerendia_claims.claim_month == kerendia_periods.period_key, "left")
    .join(
        payer_period_plan,
        (F.col("PAYER_PLAN_ID_LONG") == F.col("IMSPayerPlanId")) & (F.col("claim_month") == F.col("plan_period")),
        "left"
    )
    .drop(payer_period_plan.IMSPayerPlanId)
    .join(
        payer_status_pivoted,
        (F.col("PAYER_PLAN_ID_LONG") == payer_status_pivoted.IMSPayerPlanId) & (F.col("claim_month") == payer_status_pivoted.Period),
        "left"
    )
    .drop(payer_status_pivoted.IMSPayerPlanId)
    .drop(payer_status_pivoted.Period)
)

claims_enriched = (
    claims_with_access
    .withColumn(
        "unmatched_reason",
        F.when(F.col("period_key").isNull(), F.lit("No_Period_Match"))
         .when(F.col("plan_period").isNull(), F.lit("No_Payer_Match"))
         .when(F.col("winning_FormularyId").isNull(), F.lit("No_Drug_Row"))
         .otherwise(F.lit("Matched"))
    )
    .drop("period_key", "plan_period")
    .withColumn("kerendia_access", access_category("kerendia_status"))
    .withColumn("jardiance_access", access_category("jardiance_status"))
    .withColumn("farxiga_access", access_category("farxiga_status"))
    .withColumn("invokana_access", access_category("invokana_status"))
    .withColumn(
        "kerendia_rank",
        F.when(F.col("kerendia_status").isin("Preferred", "Generic (Preferred)"), F.lit(6))
         .when(F.col("kerendia_status") == "Covered", F.lit(5))
         .when(F.col("kerendia_status").isin("Preferred (PA/ST)", "Specialty"), F.lit(4))
         .when(F.col("kerendia_status").isin("Covered (PA/ST)", "Generic (Preferred) (PA/ST)", "Generic (PA/ST)"), F.lit(3))
         .when(F.col("kerendia_status") == "Specialty (PA/ST)", F.lit(2))
         .when(F.col("kerendia_status") == "Not Listed", F.lit(1))
         .when(F.col("kerendia_status") == "Not Covered", F.lit(0))
    )
    .withColumn(
        "jardiance_rank",
        F.when(F.col("jardiance_status").isin("Preferred", "Generic (Preferred)"), F.lit(6))
         .when(F.col("jardiance_status") == "Covered", F.lit(5))
         .when(F.col("jardiance_status").isin("Preferred (PA/ST)", "Specialty"), F.lit(4))
         .when(F.col("jardiance_status").isin("Covered (PA/ST)", "Generic (Preferred) (PA/ST)", "Generic (PA/ST)"), F.lit(3))
         .when(F.col("jardiance_status") == "Specialty (PA/ST)", F.lit(2))
         .when(F.col("jardiance_status") == "Not Listed", F.lit(1))
         .when(F.col("jardiance_status") == "Not Covered", F.lit(0))
    )
)

original_count = kerendia_claims.count()
joined_count = claims_enriched.count()
print(f"Claims before MMIT join: {original_count:,}")
print(f"Claims after MMIT join:  {joined_count:,}")
print(f"Row-count check: {'PASS' if original_count == joined_count else 'FAIL'}")

display(
    claims_enriched.groupBy("unmatched_reason")
    .count()
    .withColumn("pct", F.round(F.col("count") / F.lit(joined_count) * 100, 2))
    .orderBy(F.desc("count"))
)

display(
    claims_enriched.select(
        "CLAIM_ID", "claim_month", "PAYER_PLAN_ID", "unmatched_reason",
        "winning_OrgId", "winning_FormularyId", "kerendia_status", "jardiance_status"
    ).limit(10)
)

In [0]:
restrictions_raw = spark.sql(f"""
    SELECT FormularyId, Period, RestrictionCode
    FROM {catalog}.market_access.bronze_restrictions
    WHERE MedId = {KERENDIA_MEDID}
      AND Period BETWEEN {COVERED_START} AND {COVERED_END}
""")

restriction_flags = (
    restrictions_raw
    .groupBy("FormularyId", "Period")
    .agg(
        F.lit(1).alias("has_restriction_row"),
        F.max(F.when(F.col("RestrictionCode") == "PA", 1).otherwise(0)).alias("has_PA"),
        F.max(F.when(F.col("RestrictionCode") == "ST", 1).otherwise(0)).alias("has_ST"),
        F.max(F.when(F.col("RestrictionCode") == "QL", 1).otherwise(0)).alias("has_QL")
    )
)

pa_v2_ckd = spark.sql(f"""
    SELECT FormularyId, Period, trim(AppropriateRestrictiveStatus) AS ARS_trim
    FROM {catalog}.market_access.bronze_pa_v2
    WHERE MedId = {KERENDIA_MEDID}
      AND IndicationName = 'Chronic Kidney Disease'
      AND Period BETWEEN {COVERED_START} AND {COVERED_END}
""")

pa_v2_flags = (
    pa_v2_ckd
    .groupBy("FormularyId", "Period")
    .agg(
        F.lit(1).alias("has_pa_v2_row"),
        F.max(F.when(F.col("ARS_trim") == "R", 1).otherwise(0)).alias("ckd_pa_ars_R")
    )
)

st_ckd = spark.sql(f"""
    SELECT FormularyId, Period, StepStatus, StepProducts
    FROM {catalog}.market_access.bronze_st
    WHERE MedId = {KERENDIA_MEDID}
      AND IndicationName = 'Chronic Kidney Disease'
      AND Period BETWEEN {COVERED_START} AND {COVERED_END}
""")

st_flags = (
    st_ckd
    .groupBy("FormularyId", "Period")
    .agg(
        F.lit(1).alias("has_st_row"),
        F.max(
            F.when(
                F.upper(F.coalesce(F.col("StepProducts"), F.lit(""))).rlike("SGLT[- ]?2|JARDIANCE|FARXIGA|INVOKANA|STEGLATRO"),
                1
            ).otherwise(0)
        ).alias("step_requires_sglt2")
    )
)

pa_udf_flags = spark.sql(f"""
    SELECT FormularyId, Period, 1 AS has_pa_udf_row
    FROM {catalog}.market_access.bronze_pa_udf
    WHERE MedId = {KERENDIA_MEDID}
      AND IndicationName = 'Chronic Kidney Disease'
      AND Period BETWEEN {COVERED_START} AND {COVERED_END}
    GROUP BY FormularyId, Period
""")

hpm_raw = spark.sql(f"""
    SELECT FormularyId, Period, DerivedFieldValue, feed_date
    FROM {catalog}.market_access.bronze_derived_fields
    WHERE MedId = {KERENDIA_MEDID}
      AND DerivedFieldName = 'Bayer_ChronicKidneyDisease_HPM_Pharm'
      AND Period BETWEEN {COVERED_START} AND {COVERED_END}
""")

hpm_latest = (
    hpm_raw
    .withColumn(
        "rn",
        F.row_number().over(Window.partitionBy("FormularyId", "Period").orderBy(F.col("feed_date").desc(), F.col("DerivedFieldValue")))
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumnRenamed("DerivedFieldValue", "ckd_hpm_value")
    .withColumn(
        "ckd_hpm_group",
        F.when(F.col("ckd_hpm_value").isin("Preferred -Unrestricted", "Covered -Unrestricted"), "Unrestricted")
         .when(F.col("ckd_hpm_value").contains("PA to Label"), "PA_to_Label")
         .when(F.col("ckd_hpm_value").contains("Step[s]"), "Step")
         .when(F.col("ckd_hpm_value").contains("PA Unspecified"), "PA_Unspecified")
         .when(F.col("ckd_hpm_value").contains("PA+"), "PA_Plus")
         .when(F.col("ckd_hpm_value") == "APR", "APR")
         .when(F.col("ckd_hpm_value") == "Medical Exception", "Medical_Exception")
         .when((F.col("ckd_hpm_value").isNull()) | (F.col("ckd_hpm_value") == "Unknown"), "Unknown")
         .otherwise("Other")
    )
)

status_monthly = (
    status_deduped
    .filter(F.col("Period").between(COVERED_START, 202609))
    .withColumn("status_rank", F.when(F.col("status_rollup").isin("Preferred", "Generic (Preferred)"), F.lit(6))
                               .when(F.col("status_rollup") == "Covered", F.lit(5))
                               .when(F.col("status_rollup").isin("Preferred (PA/ST)", "Specialty"), F.lit(4))
                               .when(F.col("status_rollup").isin("Covered (PA/ST)", "Generic (Preferred) (PA/ST)", "Generic (PA/ST)"), F.lit(3))
                               .when(F.col("status_rollup") == "Specialty (PA/ST)", F.lit(2))
                               .when(F.col("status_rollup") == "Not Listed", F.lit(1))
                               .when(F.col("status_rollup") == "Not Covered", F.lit(0)))
)

change_window = Window.partitionBy("FormularyId", "drug_label").orderBy("Period")
rolling3_window = change_window.rowsBetween(-2, 0)
status_changes_monthly = (
    status_monthly
    .withColumn("prev_rank", F.lag("status_rank").over(change_window))
    .withColumn("improved_mom", F.when(F.col("prev_rank").isNotNull() & (F.col("status_rank") > F.col("prev_rank")), 1).otherwise(0))
    .withColumn("worsened_mom", F.when(F.col("prev_rank").isNotNull() & (F.col("status_rank") < F.col("prev_rank")), 1).otherwise(0))
    .withColumn("changed_mom", F.when(F.col("prev_rank").isNotNull() & (F.col("status_rank") != F.col("prev_rank")), 1).otherwise(0))
    .withColumn("improved_l3m", F.max("improved_mom").over(rolling3_window))
    .withColumn("worsened_l3m", F.max("worsened_mom").over(rolling3_window))
    .withColumn("changed_l3m", F.max("changed_mom").over(rolling3_window))
)

change_flags = (
    status_changes_monthly
    .filter(F.col("drug_label").isin("kerendia", "farxiga", "jardiance"))
    .groupBy("FormularyId", "Period")
    .agg(
        F.max(F.when(F.col("drug_label") == "kerendia", F.col("improved_l3m"))).alias("kerendia_improved_l3m"),
        F.max(F.when(F.col("drug_label") == "kerendia", F.col("worsened_l3m"))).alias("kerendia_worsened_l3m"),
        F.max(F.when(F.col("drug_label") == "kerendia", F.col("changed_l3m"))).alias("kerendia_changed_l3m"),
        F.max(F.when(F.col("drug_label") == "farxiga", F.col("worsened_l3m"))).alias("farxiga_worsened_l3m"),
        F.max(F.when(F.col("drug_label") == "jardiance", F.col("worsened_l3m"))).alias("jardiance_worsened_l3m")
    )
)

base_claims_window = claims_enriched.filter(F.col("claim_month").between(COVERED_START, COVERED_END))
base_n = base_claims_window.count()

claims_feature_base = base_claims_window
for name, lookup in [
    ("restriction_flags", restriction_flags),
    ("pa_v2_flags", pa_v2_flags),
    ("st_flags", st_flags),
    ("pa_udf_flags", pa_udf_flags),
    ("hpm_latest", hpm_latest.select("FormularyId", "Period", "ckd_hpm_value", "ckd_hpm_group")),
    ("change_flags", change_flags)
]:
    claims_feature_base = claims_feature_base.join(
        lookup,
        (claims_feature_base.winning_FormularyId == lookup.FormularyId) & (claims_feature_base.claim_month == lookup.Period),
        "left"
    ).drop(lookup.FormularyId).drop(lookup.Period)
    after_n = claims_feature_base.count()
    print(f"Join {name}: {after_n:,} rows ({'PASS' if after_n == base_n else 'FAIL'})")

claims_feature_base = (
    claims_feature_base
    .withColumn("has_restriction_row", F.coalesce(F.col("has_restriction_row"), F.lit(0)))
    .withColumn("has_PA", F.coalesce(F.col("has_PA"), F.lit(0)))
    .withColumn("has_ST", F.coalesce(F.col("has_ST"), F.lit(0)))
    .withColumn("has_QL", F.coalesce(F.col("has_QL"), F.lit(0)))
    .withColumn("has_pa_v2_row", F.coalesce(F.col("has_pa_v2_row"), F.lit(0)))
    .withColumn("ckd_pa_ars_R", F.coalesce(F.col("ckd_pa_ars_R"), F.lit(0)))
    .withColumn("has_st_row", F.coalesce(F.col("has_st_row"), F.lit(0)))
    .withColumn("step_requires_sglt2", F.coalesce(F.col("step_requires_sglt2"), F.lit(0)))
    .withColumn("has_pa_udf_row", F.coalesce(F.col("has_pa_udf_row"), F.lit(0)))
    .withColumn("kerendia_improved_l3m", F.coalesce(F.col("kerendia_improved_l3m"), F.lit(0)))
    .withColumn("kerendia_worsened_l3m", F.coalesce(F.col("kerendia_worsened_l3m"), F.lit(0)))
    .withColumn("kerendia_changed_l3m", F.coalesce(F.col("kerendia_changed_l3m"), F.lit(0)))
    .withColumn("farxiga_worsened_l3m", F.coalesce(F.col("farxiga_worsened_l3m"), F.lit(0)))
    .withColumn("jardiance_worsened_l3m", F.coalesce(F.col("jardiance_worsened_l3m"), F.lit(0)))
)

print(f"Claims in feature base: {claims_feature_base.count():,}")
display(
    claims_feature_base.select(
        "CLAIM_ID", "claim_month", "unmatched_reason", "winning_FormularyId",
        "has_PA", "has_ST", "ckd_pa_ars_R", "step_requires_sglt2", "ckd_hpm_group"
    ).limit(10)
)

In [0]:
section1_monthly = (
    claims_enriched
    .groupBy("claim_month")
    .agg(
        F.count("*").alias("claims"),
        F.sum(F.when(F.col("unmatched_reason") == "Matched", 1).otherwise(0)).alias("matched_claims"),
        F.sum(F.when(F.col("unmatched_reason") == "No_Period_Match", 1).otherwise(0)).alias("No_Period_Match"),
        F.sum(F.when(F.col("unmatched_reason") == "No_Payer_Match", 1).otherwise(0)).alias("No_Payer_Match"),
        F.sum(F.when(F.col("unmatched_reason") == "No_Drug_Row", 1).otherwise(0)).alias("No_Drug_Row")
    )
    .withColumn("unmatched_rate_pct", F.round((F.col("claims") - F.col("matched_claims")) / F.col("claims") * 100, 2))
    .withColumn("No_Period_Match_pct", F.round(F.col("No_Period_Match") / F.col("claims") * 100, 2))
    .withColumn("No_Payer_Match_pct", F.round(F.col("No_Payer_Match") / F.col("claims") * 100, 2))
    .withColumn("No_Drug_Row_pct", F.round(F.col("No_Drug_Row") / F.col("claims") * 100, 2))
    .orderBy("claim_month")
)

display(section1_monthly)

covered_window_claims = claims_enriched.filter(F.col("claim_month").between(COVERED_START, COVERED_END))
window_total = covered_window_claims.count()
window_reason_counts = {
    row["unmatched_reason"]: row["count"]
    for row in covered_window_claims.groupBy("unmatched_reason").count().collect()
}
window_unmatched = window_total - window_reason_counts.get("Matched", 0)

section1_top_unmatched = (
    covered_window_claims
    .filter(F.col("unmatched_reason") != "Matched")
    .groupBy("PAYER_PLAN_ID")
    .agg(F.count("*").alias("claims"))
    .withColumn("claim_share_pct", F.round(F.col("claims") / F.lit(window_unmatched) * 100, 2))
    .orderBy(F.desc("claims"), F.asc("PAYER_PLAN_ID"))
    .limit(15)
)

display(section1_top_unmatched)

monthly_pdf = section1_monthly.toPandas()
monthly_pdf = monthly_pdf[[
    "claim_month", "claims", "matched_claims", "unmatched_rate_pct",
    "No_Period_Match_pct", "No_Payer_Match_pct", "No_Drug_Row_pct"
]]
monthly_pdf.columns = [
    "claim_month", "claims", "matched_claims", "unmatched_rate_pct",
    "No_Period_Match_pct", "No_Payer_Match_pct", "No_Drug_Row_pct"
]
top_unmatched_pdf = section1_top_unmatched.toPandas()

section1_md = f"""
* Covered-window claims (`202510`-`202608`): **{window_total:,}**
* Covered-window unmatched claims: **{window_unmatched:,}** (**{pct(window_unmatched, window_total):.2f}%**)
* Covered-window split: `No_Payer_Match` = **{window_reason_counts.get('No_Payer_Match', 0):,}** (**{pct(window_reason_counts.get('No_Payer_Match', 0), window_total):.2f}%**), `No_Drug_Row` = **{window_reason_counts.get('No_Drug_Row', 0):,}** (**{pct(window_reason_counts.get('No_Drug_Row', 0), window_total):.2f}%**)
* `No_Period_Match` is zero inside the covered window by construction.

Unmatched-rate trend by claim month:

{pdf_to_md_table(monthly_pdf)}

Top 15 unmatched `PAYER_PLAN_ID` values inside `202510`-`202608` (share of unmatched claims in-window):

{pdf_to_md_table(top_unmatched_pdf)}
""".strip()

print("SECTION1_MD_START")
print(section1_md)
print("SECTION1_MD_END")

In [0]:
matched_claims_window = claims_feature_base.filter(F.col("unmatched_reason") == "Matched")
matched_total = matched_claims_window.count()

coverage_rows = [
    ("bronze_pa_v2", matched_claims_window.filter(F.col("has_pa_v2_row") == 1).count()),
    ("bronze_st", matched_claims_window.filter(F.col("has_st_row") == 1).count()),
    ("bronze_pa_udf", matched_claims_window.filter(F.col("has_pa_udf_row") == 1).count()),
    ("bronze_derived_fields HPM", matched_claims_window.filter(F.col("ckd_hpm_value").isNotNull()).count()),
    ("bronze_restrictions", matched_claims_window.filter(F.col("has_restriction_row") == 1).count()),
]
coverage_pdf = pd.DataFrame(
    [{
        "table": name,
        "matched_claims_with_row": cnt,
        "claim_weighted_coverage_pct": float(np.round((cnt / matched_total) * 100, 2)) if matched_total else None,
    } for name, cnt in coverage_rows]
)
print(coverage_pdf.to_string(index=False))

preferred_claims = matched_claims_window.filter(F.col("kerendia_status") == "Preferred")
preferred_total = preferred_claims.count()
preferred_mix_pdf = pd.DataFrame([
    ("Any restriction row", preferred_claims.filter(F.col("has_restriction_row") == 1).count()),
    ("PA", preferred_claims.filter(F.col("has_PA") == 1).count()),
    ("ST", preferred_claims.filter(F.col("has_ST") == 1).count()),
    ("QL", preferred_claims.filter(F.col("has_QL") == 1).count()),
    ("QL only", preferred_claims.filter((F.col("has_QL") == 1) & (F.col("has_PA") == 0) & (F.col("has_ST") == 0)).count()),
], columns=["restriction_mix", "claims"])
preferred_mix_pdf["pct_of_preferred_claims"] = preferred_mix_pdf["claims"].map(lambda x: float(np.round((x / preferred_total) * 100, 2)) if preferred_total else None)
print(preferred_mix_pdf.to_string(index=False))

section3_md = f"""
* Matched Kerendia claims in `202510`-`202608`: **{matched_total:,}** (denominator for all percentages below).

Claim-weighted coverage on the winning `FormularyId + Period`:

{pdf_to_md_table(coverage_pdf)}

Restriction-code mix for claims on raw `Preferred` Kerendia formularies (`n = {preferred_total:,}` claims):

{pdf_to_md_table(preferred_mix_pdf)}
""".strip()

print("SECTION3_MD_START")
print(section3_md)
print("SECTION3_MD_END")

In [0]:
section4_claims = claims_feature_base.filter(
    (F.col("unmatched_reason") == "Matched") &
    (F.col("claim_month").between(SECTION4_START, SECTION4_END))
)
section4_total = section4_claims.count()

xtab = (
    section4_claims
    .groupBy("kerendia_access", "ckd_hpm_group")
    .count()
    .withColumn("claim_share_pct", F.round(F.col("count") / F.lit(section4_total) * 100, 2))
    .orderBy("kerendia_access", F.desc("count"))
)

display(xtab)
xtab_pdf = xtab.toPandas()

change_exposure = (
    section4_claims
    .groupBy("claim_month")
    .agg(
        F.round(F.avg(F.col("kerendia_changed_l3m")) * 100, 2).alias("kerendia_changed_l3m_pct"),
        F.round(F.avg(F.col("kerendia_improved_l3m")) * 100, 2).alias("kerendia_improved_l3m_pct"),
        F.round(F.avg(F.col("kerendia_worsened_l3m")) * 100, 2).alias("kerendia_worsened_l3m_pct"),
        F.round(F.avg(F.col("farxiga_worsened_l3m")) * 100, 2).alias("farxiga_worsened_l3m_pct")
    )
    .orderBy("claim_month")
)

display(change_exposure)
change_exposure_pdf = change_exposure.toPandas()

section4_md = f"""
* Matched claims in `202601`-`202608`: **{section4_total:,}**

Claim-weighted cross-tab of `kerendia_access x kerendia_ckd_hpm_group`:

{pdf_to_md_table(xtab_pdf)}
""".strip()

section6_md = f"""
Claim-weighted exposure to L3M status changes on the winning formulary (`202601`-`202608`):

{pdf_to_md_table(change_exposure_pdf)}
""".strip()

print("SECTION4_MD_START")
print(section4_md)
print("SECTION4_MD_END")
print("SECTION6_MD_START")
print(section6_md)
print("SECTION6_MD_END")

In [0]:
patient_first_month = kerendia_claims.groupBy("PATIENT_ID").agg(F.min("claim_month").alias("patient_first_month"))
claims_with_first = kerendia_claims.join(patient_first_month, on="PATIENT_ID", how="left")

provider_month_outcomes = (
    claims_with_first
    .groupBy("PROVIDER_ID", "claim_month")
    .agg(
        F.countDistinct("CLAIM_ID").alias("month_trx"),
        F.countDistinct(F.when(F.col("patient_first_month") == F.col("claim_month"), F.col("PATIENT_ID"))).alias("month_nbrx_patients")
    )
)

provider_month_features = (
    claims_feature_base
    .filter(F.col("claim_month").between(COVERED_START, SECTION5_END))
    .groupBy("PROVIDER_ID", "claim_month")
    .agg(
        F.countDistinct("CLAIM_ID").alias("current_month_trx"),
        F.avg(F.when(F.col("kerendia_access") == "Covered", 1).otherwise(0)).alias("share_kerendia_Covered"),
        F.avg(F.when(F.col("kerendia_access") == "Covered_with_PA_ST", 1).otherwise(0)).alias("share_kerendia_Covered_with_PA_ST"),
        F.avg(F.when(F.col("kerendia_access") == "Not_Covered", 1).otherwise(0)).alias("share_kerendia_Not_Covered"),
        F.avg(F.when(F.col("kerendia_status").startswith("Preferred"), 1).otherwise(0)).alias("share_kerendia_Preferred"),
        F.avg(F.when(F.col("kerendia_rank").isNotNull() & F.col("jardiance_rank").isNotNull() & (F.col("kerendia_rank") < F.col("jardiance_rank")), 1).otherwise(0)).alias("share_kerendia_worse_than_jardiance"),
        F.avg(F.col("has_PA")).alias("share_has_PA"),
        F.avg(F.col("has_ST")).alias("share_has_ST"),
        F.avg(F.col("step_requires_sglt2")).alias("share_step_requires_sglt2"),
        F.avg(F.col("ckd_pa_ars_R")).alias("share_ckd_pa_ars_R"),
        F.avg(F.when(F.col("ckd_hpm_group") == "Unrestricted", 1).otherwise(0)).alias("share_hpm_unrestricted"),
        F.avg(F.when(F.col("ckd_hpm_group") == "PA_to_Label", 1).otherwise(0)).alias("share_hpm_pa_to_label"),
        F.avg(F.when(F.col("ckd_hpm_group") == "Step", 1).otherwise(0)).alias("share_hpm_step"),
        F.avg(F.col("kerendia_improved_l3m")).alias("share_kerendia_improved_l3m"),
        F.avg(F.col("kerendia_worsened_l3m")).alias("share_kerendia_worsened_l3m"),
        F.avg(F.when(F.col("Channel") == "Medicare", 1).otherwise(0)).alias("share_medicare"),
        F.avg(F.when(F.col("Channel") == "Commercial", 1).otherwise(0)).alias("share_commercial"),
        F.avg(F.when(F.col("PBMName").isin(TOP3_PBMS), 1).otherwise(0)).alias("share_top3_pbm"),
        F.avg(F.when(F.col("kerendia_status").isNotNull(), 1).otherwise(0)).alias("fill_access"),
        F.avg(F.when(F.col("kerendia_status").isNotNull() & F.col("jardiance_status").isNotNull(), 1).otherwise(0)).alias("fill_vs_jardiance"),
        F.avg(F.when(F.col("has_restriction_row") == 1, 1).otherwise(0)).alias("fill_restrictions"),
        F.avg(F.when(F.col("has_st_row") == 1, 1).otherwise(0)).alias("fill_st"),
        F.avg(F.when(F.col("has_pa_v2_row") == 1, 1).otherwise(0)).alias("fill_pa_v2"),
        F.avg(F.when(F.col("ckd_hpm_value").isNotNull(), 1).otherwise(0)).alias("fill_hpm"),
        F.avg(F.when(F.col("winning_FormularyId").isNotNull(), 1).otherwise(0)).alias("fill_plan"),
        F.avg(F.when(F.col("winning_FormularyId").isNotNull(), 1).otherwise(0)).alias("fill_change")
    )
    .withColumn("month_date", month_to_date("claim_month"))
    .withColumn("next_month", F.date_format(F.add_months(F.col("month_date"), 1), "yyyyMM").cast("int"))
)

provider_month_model = (
    provider_month_features
    .join(
        provider_month_outcomes.select(
            F.col("PROVIDER_ID").alias("target_PROVIDER_ID"),
            F.col("claim_month").alias("target_month"),
            F.col("month_nbrx_patients").alias("next_month_nbrx_patients"),
            F.col("month_trx").alias("next_month_trx")
        ),
        (provider_month_features.PROVIDER_ID == F.col("target_PROVIDER_ID")) & (provider_month_features.next_month == F.col("target_month")),
        "left"
    )
    .drop("target_PROVIDER_ID", "target_month")
    .fillna({"next_month_nbrx_patients": 0, "next_month_trx": 0})
    .withColumn("next_month_nbrx_flag", F.when(F.col("next_month_nbrx_patients") > 0, 1).otherwise(0))
)

model_pdf = provider_month_model.select(
    "PROVIDER_ID", "claim_month", "current_month_trx", "next_month_nbrx_flag", "next_month_trx",
    "share_kerendia_Covered", "share_kerendia_Covered_with_PA_ST", "share_kerendia_Not_Covered",
    "share_kerendia_Preferred", "share_kerendia_worse_than_jardiance", "share_has_PA", "share_has_ST",
    "share_step_requires_sglt2", "share_ckd_pa_ars_R", "share_hpm_unrestricted", "share_hpm_pa_to_label",
    "share_hpm_step", "share_kerendia_improved_l3m", "share_kerendia_worsened_l3m", "share_medicare",
    "share_commercial", "share_top3_pbm", "fill_access", "fill_vs_jardiance", "fill_restrictions",
    "fill_st", "fill_pa_v2", "fill_hpm", "fill_plan", "fill_change"
).toPandas()

model_pdf["volume_tercile"] = pd.qcut(model_pdf["current_month_trx"].rank(method="first"), 3, labels=["low", "mid", "high"])
overall_rate = model_pdf["next_month_nbrx_flag"].mean()

feature_fill_map = {
    "share_kerendia_Covered": "fill_access",
    "share_kerendia_Covered_with_PA_ST": "fill_access",
    "share_kerendia_Not_Covered": "fill_access",
    "share_kerendia_Preferred": "fill_access",
    "share_kerendia_worse_than_jardiance": "fill_vs_jardiance",
    "share_has_PA": "fill_restrictions",
    "share_has_ST": "fill_restrictions",
    "share_step_requires_sglt2": "fill_st",
    "share_ckd_pa_ars_R": "fill_pa_v2",
    "share_hpm_unrestricted": "fill_hpm",
    "share_hpm_pa_to_label": "fill_hpm",
    "share_hpm_step": "fill_hpm",
    "share_kerendia_improved_l3m": "fill_change",
    "share_kerendia_worsened_l3m": "fill_change",
    "share_medicare": "fill_plan",
    "share_commercial": "fill_plan",
    "share_top3_pbm": "fill_plan",
}

features = list(feature_fill_map.keys())


def auc_from_scores(y_true, scores):
    work = pd.DataFrame({"y": y_true, "s": scores}).dropna()
    if work.empty or work["y"].nunique() < 2 or work["s"].nunique() < 2:
        return np.nan
    work["rank"] = work["s"].rank(method="average")
    n_pos = work["y"].sum()
    n_neg = len(work) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    rank_sum_pos = work.loc[work["y"] == 1, "rank"].sum()
    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def top_bottom_lift(df, feature, overall):
    work = df[[feature, "next_month_nbrx_flag"]].dropna().copy()
    if work.empty or work[feature].nunique() < 2 or overall == 0:
        return np.nan, np.nan
    if work[feature].nunique() <= 2:
        work["bin"] = (work[feature] > work[feature].min()).astype(int) + 1
    else:
        work["bin"] = pd.qcut(work[feature].rank(method="first"), 5, labels=False) + 1
    bin_rate = work.groupby("bin")["next_month_nbrx_flag"].mean()
    top_lift = bin_rate.loc[bin_rate.index.max()] / overall
    bottom_lift = bin_rate.loc[bin_rate.index.min()] / overall
    return float(top_lift), float(bottom_lift)


rows = []
for feature in features:
    fill_rate = float(model_pdf[feature_fill_map[feature]].mean() * 100)
    auc = auc_from_scores(model_pdf["next_month_nbrx_flag"], model_pdf[feature])
    top_lift, bottom_lift = top_bottom_lift(model_pdf, feature, overall_rate)
    spearman = model_pdf[[feature, "next_month_trx"]].corr(method="spearman").iloc[0, 1]
    tercile_lifts = []
    for tercile in ["low", "mid", "high"]:
        subset = model_pdf[model_pdf["volume_tercile"] == tercile]
        overall_subset = subset["next_month_nbrx_flag"].mean()
        t_lift, _ = top_bottom_lift(subset, feature, overall_subset)
        tercile_lifts.append(None if pd.isna(t_lift) else float(np.round(float(t_lift), 3)))
    rows.append({
        "Feature": feature,
        "Fill rate": float(np.round(fill_rate, 2)),
        "AUC": None if pd.isna(auc) else float(np.round(float(auc), 4)),
        "Top-bin lift": None if pd.isna(top_lift) else float(np.round(float(top_lift), 3)),
        "Bottom-bin lift": None if pd.isna(bottom_lift) else float(np.round(float(bottom_lift), 3)),
        "Lift within volume terciles (low / mid / high)": " / ".join(["NA" if x is None else str(x) for x in tercile_lifts]),
        "Spearman_next_month_trx": None if pd.isna(spearman) else float(np.round(float(spearman), 4)),
    })

section5_results_pdf = pd.DataFrame(rows).sort_values(["AUC", "Top-bin lift"], ascending=[False, False])
display(spark.createDataFrame(section5_results_pdf.fillna("NA")))

section5_md = pdf_to_md_table(section5_results_pdf[[
    "Feature", "Fill rate", "AUC", "Top-bin lift", "Bottom-bin lift", "Lift within volume terciles (low / mid / high)"
]])
section5_notes = f"Overall next-month NBRx flag rate: {overall_rate * 100:.2f}% across {len(model_pdf):,} HCP-month rows."

print("SECTION5_MD_START")
print(section5_notes)
print(section5_md)
print("SECTION5_MD_END")

In [0]:
# Section 5b: does access-feature signal on next-month NBRx depend on how exposed the HCP is to restrictions?
# HCPs are split into terciles by their number of restricted Kerendia claims (PA, ST or QL on the winning formulary)
# over 202510-202607. AUC of each access feature vs next_month_nbrx_flag is then computed within each tercile.
# Requires: claims_feature_base (cell 6), model_pdf / features / auc_from_scores / top_bottom_lift (Section 5 cell), pdf_to_md_table.

sec5_claims = claims_feature_base.filter(F.col("claim_month").between(COVERED_START, SECTION5_END))

is_restricted = (F.col("has_PA") == 1) | (F.col("has_ST") == 1) | (F.col("has_QL") == 1)
is_pa_st = (F.col("has_PA") == 1) | (F.col("has_ST") == 1)

hcp_restricted_pdf = (
    sec5_claims
    .groupBy("PROVIDER_ID")
    .agg(
        F.countDistinct("CLAIM_ID").alias("hcp_claims"),
        F.countDistinct(F.when(F.col("unmatched_reason") == "Matched", F.col("CLAIM_ID"))).alias("hcp_matched_claims"),
        F.countDistinct(F.when(is_restricted, F.col("CLAIM_ID"))).alias("hcp_restricted_claims"),
        F.countDistinct(F.when(is_pa_st, F.col("CLAIM_ID"))).alias("hcp_pa_st_claims"),
    )
    .toPandas()
)

# Extra feature: share of QL claims per HCP-month (not in the Section 5 feature set)
ql_share_pdf = (
    sec5_claims
    .groupBy("PROVIDER_ID", "claim_month")
    .agg(
        F.avg(F.col("has_QL")).alias("share_has_QL"),
        F.avg(F.when(is_restricted, 1).otherwise(0)).alias("share_any_restriction"),
    )
    .toPandas()
)


def assign_terciles(values):
    """Terciles on restricted-claim counts. Falls back to Low = 0 restricted claims and a median split of the rest
    when ties at zero make quantile edges non-unique."""
    try:
        return pd.qcut(values, 3, labels=["Low", "Medium", "High"]).astype(str), "quantile"
    except ValueError:
        out = pd.Series("Low", index=values.index)
        pos = values > 0
        med = values[pos].median()
        out[pos & (values <= med)] = "Medium"
        out[pos & (values > med)] = "High"
        return out, "zero_vs_median_split"


hcp_restricted_pdf["restricted_tercile"], tercile_method = assign_terciles(hcp_restricted_pdf["hcp_restricted_claims"])
print(f"Tercile method: {tercile_method}")

strat_pdf = (
    model_pdf
    .merge(ql_share_pdf, on=["PROVIDER_ID", "claim_month"], how="left")
    .merge(hcp_restricted_pdf[["PROVIDER_ID", "hcp_restricted_claims", "hcp_pa_st_claims", "restricted_tercile"]], on="PROVIDER_ID", how="left")
)
print(f"HCP-month rows: {len(strat_pdf):,} (Section 5 base: {len(model_pdf):,})")

strat_features = features + ["share_has_QL", "share_any_restriction"]
terciles = ["Low", "Medium", "High"]

# Tercile profile
profile_rows = []
for t in terciles:
    hcps = hcp_restricted_pdf[hcp_restricted_pdf["restricted_tercile"] == t]
    rows_t = strat_pdf[strat_pdf["restricted_tercile"] == t]
    profile_rows.append({
        "Tercile": t,
        "HCPs": int(len(hcps)),
        "Restricted claims per HCP (min-max)": f"{int(hcps['hcp_restricted_claims'].min())}-{int(hcps['hcp_restricted_claims'].max())}",
        "Median restricted claims": float(hcps["hcp_restricted_claims"].median()),
        "Median total claims": float(hcps["hcp_claims"].median()),
        "Restricted share of claims %": float(np.round(hcps["hcp_restricted_claims"].sum() / max(hcps["hcp_claims"].sum(), 1) * 100, 2)),
        "HCP-months": int(len(rows_t)),
        "Next-month NBRx rate %": float(np.round(rows_t["next_month_nbrx_flag"].mean() * 100, 2)),
    })
tercile_profile_pdf = pd.DataFrame(profile_rows)
display(spark.createDataFrame(tercile_profile_pdf))

# AUC per feature: overall and within each tercile
auc_rows = []
for feat in strat_features:
    row = {"Feature": feat}
    overall_auc = auc_from_scores(strat_pdf["next_month_nbrx_flag"], strat_pdf[feat])
    row["AUC overall"] = None if pd.isna(overall_auc) else float(np.round(overall_auc, 4))
    for t in terciles:
        sub = strat_pdf[strat_pdf["restricted_tercile"] == t]
        a = auc_from_scores(sub["next_month_nbrx_flag"], sub[feat])
        row[f"AUC {t}"] = None if pd.isna(a) else float(np.round(a, 4))
    t_vals = [row[f"AUC {t}"] for t in terciles if row[f"AUC {t}"] is not None]
    if t_vals:
        best_t = max(terciles, key=lambda t: -1 if row[f"AUC {t}"] is None else abs(row[f"AUC {t}"] - 0.5))
        row["Strongest tercile"] = best_t
        row["Max distance from 0.5"] = float(np.round(max(abs(v - 0.5) for v in t_vals), 4))
        row["AUC range across terciles"] = float(np.round(max(t_vals) - min(t_vals), 4))
    else:
        row["Strongest tercile"] = "NA"
        row["Max distance from 0.5"] = None
        row["AUC range across terciles"] = None
    auc_rows.append(row)

section5b_auc_pdf = pd.DataFrame(auc_rows).sort_values("Max distance from 0.5", ascending=False)
display(spark.createDataFrame(section5b_auc_pdf.fillna("NA").astype(str)))

section5b_profile_md = pdf_to_md_table(tercile_profile_pdf)
section5b_auc_md = pdf_to_md_table(section5b_auc_pdf.fillna("NA"))

print("SECTION5B_MD_START")
print(f"Tercile method: {tercile_method}. Restricted claim = Kerendia claim whose winning formulary has a PA, ST or QL row in bronze_restrictions.")
print(section5b_profile_md)
print()
print(section5b_auc_md)
print("SECTION5B_MD_END")

In [0]:
# Section 5c: AUC of access features vs next-month NBRx within HCP terciles of Kerendia NBRx volume.
# Primary strata: HCP NBRx in a PRE-window baseline (202504-202509), so strata do not overlap the target months.
#   (202504 start gives >= 7 months of lookback from 202409 for the new-patient definition.)
# Sensitivity strata: HCP NBRx inside the feature window (202510-202607); overlaps the target, so read with caution.
# Requires: kerendia_claims, claims_feature_base, model_pdf, features, auc_from_scores, pdf_to_md_table (earlier cells).

BASELINE_START, BASELINE_END = 202504, 202509

patient_first = kerendia_claims.groupBy("PATIENT_ID").agg(F.min("claim_month").alias("first_month"))
new_pat_claims = (
    kerendia_claims.join(patient_first, "PATIENT_ID")
    .filter(F.col("claim_month") == F.col("first_month"))
    .select("PROVIDER_ID", "PATIENT_ID", "claim_month").distinct()
)

nbrx_base_pdf = (new_pat_claims.filter(F.col("claim_month").between(BASELINE_START, BASELINE_END))
    .groupBy("PROVIDER_ID").agg(F.countDistinct("PATIENT_ID").alias("nbrx_baseline")).toPandas())
nbrx_win_pdf = (new_pat_claims.filter(F.col("claim_month").between(COVERED_START, SECTION5_END))
    .groupBy("PROVIDER_ID").agg(F.countDistinct("PATIENT_ID").alias("nbrx_window")).toPandas())

sec5_claims = claims_feature_base.filter(F.col("claim_month").between(COVERED_START, SECTION5_END))
is_restricted = (F.col("has_PA") == 1) | (F.col("has_ST") == 1) | (F.col("has_QL") == 1)
extra_pdf = (sec5_claims.groupBy("PROVIDER_ID", "claim_month").agg(
    F.avg(F.col("has_QL")).alias("share_has_QL"),
    F.avg(F.when(is_restricted, 1).otherwise(0)).alias("share_any_restriction")).toPandas())

hcps_pdf = pd.DataFrame({"PROVIDER_ID": model_pdf["PROVIDER_ID"].unique()})
hcps_pdf = hcps_pdf.merge(nbrx_base_pdf, on="PROVIDER_ID", how="left").merge(nbrx_win_pdf, on="PROVIDER_ID", how="left")
hcps_pdf[["nbrx_baseline", "nbrx_window"]] = hcps_pdf[["nbrx_baseline", "nbrx_window"]].fillna(0)
hcp_claims_pdf = model_pdf.groupby("PROVIDER_ID")["current_month_trx"].sum().rename("hcp_claims").reset_index()
hcps_pdf = hcps_pdf.merge(hcp_claims_pdf, on="PROVIDER_ID", how="left")


def assign_terciles(values):
    """Quantile terciles; if ties at 0 collapse edges, Low = 0 and the positives are split at the median."""
    try:
        return pd.qcut(values, 3, labels=["Low", "Medium", "High"]).astype(str), "quantile"
    except ValueError:
        out = pd.Series("Low", index=values.index)
        pos = values > 0
        med = values[pos].median()
        out[pos & (values <= med)] = "Medium"
        out[pos & (values > med)] = "High"
        return out, "zero_vs_median_split"


strat_features = features + ["share_has_QL", "share_any_restriction"]
terciles = ["Low", "Medium", "High"]
base_pdf = model_pdf.merge(extra_pdf, on=["PROVIDER_ID", "claim_month"], how="left")


def run_strata(strat_col, label):
    h = hcps_pdf.copy()
    h["nbrx_tercile"], method = assign_terciles(h[strat_col])
    df = base_pdf.merge(h[["PROVIDER_ID", "nbrx_tercile"]], on="PROVIDER_ID", how="left")
    prof = []
    for t in terciles:
        ht = h[h["nbrx_tercile"] == t]
        dt = df[df["nbrx_tercile"] == t]
        prof.append({
            "Tercile": t, "HCPs": int(len(ht)),
            "NBRx per HCP (min-max)": f"{int(ht[strat_col].min())}-{int(ht[strat_col].max())}" if len(ht) else "NA",
            "Median NBRx": float(ht[strat_col].median()) if len(ht) else None,
            "Median total claims (window)": float(ht["hcp_claims"].median()) if len(ht) else None,
            "HCP-months": int(len(dt)),
            "Next-month NBRx rate %": float(np.round(dt["next_month_nbrx_flag"].mean() * 100, 2)) if len(dt) else None,
        })
    rows = []
    for feat in strat_features:
        r = {"Feature": feat}
        a_all = auc_from_scores(df["next_month_nbrx_flag"], df[feat])
        r["AUC overall"] = None if pd.isna(a_all) else round(a_all, 4)
        for t in terciles:
            sub = df[df["nbrx_tercile"] == t]
            a = auc_from_scores(sub["next_month_nbrx_flag"], sub[feat])
            r[f"AUC {t}"] = None if pd.isna(a) else round(a, 4)
        vals = {t: r[f"AUC {t}"] for t in terciles if r[f"AUC {t}"] is not None}
        if vals:
            best = max(vals, key=lambda t: abs(vals[t] - 0.5))
            r["Strongest tercile"] = best
            r["Max distance from 0.5"] = round(abs(vals[best] - 0.5), 4)
            r["AUC range across terciles"] = round(max(vals.values()) - min(vals.values()), 4)
        rows.append(r)
    auc_pdf = pd.DataFrame(rows).sort_values("Max distance from 0.5", ascending=False)
    print(f"{label}_MD_START")
    print(f"Strata column: {strat_col}. Tercile method: {method}.")
    print(pdf_to_md_table(pd.DataFrame(prof)))
    print()
    print(pdf_to_md_table(auc_pdf.fillna("NA")))
    print(f"{label}_MD_END")
    return auc_pdf


section5c_primary = run_strata("nbrx_baseline", "SECTION5C_PRIMARY")
section5c_sensitivity = run_strata("nbrx_window", "SECTION5C_SENSITIVITY")
display(spark.createDataFrame(section5c_primary.fillna("NA").astype(str)))